# VIKAAS — emotional Hindi voices **v2** (2 clicks)
**What's v2:** the spoken character-tags ("मम्मी बोलीं —") are REMOVED from the lines (the video already shows who's talking), and the narrator rates are tuned to the real rhythm grid. v1 voices were beautiful, v2 fits the film.

1. Run **Cell A** → it speaks all 8 lines + prints a FIT CHECK (must say ✅ on all rows).
2. Run **Cell C** → downloads `vo_out.zip`. Send it to Arena. Done.

*(Cell B stays the optional fully-open-source fallback — skip unless asked.)*

In [ ]:
# ===== CELL A v2 — edge-tts, tagless lines, grid-matched pacing =====
!pip -q install edge-tts nest-asyncio mutagen
import os, asyncio, edge_tts, nest_asyncio
nest_asyncio.apply()
os.makedirs('/content/vo_out', exist_ok=True)

LINES = {
  'vo1_pov':       'तुम घर का इकलौता ई-वेस्ट वॉरियर हो।',
  'vo2_mummy':     'वो चार्जर मत फेंकना। कभी काम आएगा।',
  'vo3_narrator1': 'वो चार्जर दो हज़ार चौदह से एक बार भी काम नहीं आया।',
  'vo4_papa':      'पुराने फ़ोन में सोना होता है बेटा।',
  'vo5_narrator2': 'तक़रीबन शून्य दशमलव शून्य तीन ग्राम। चिंगम भी महँगी है।',
  'vo6_calc':      'मिनिमम पाँच सौ किलो। मेरे पास — सवा किलो। शॉर्ट — सिर्फ़ चार सौ अट्ठानबे दशमलव छह किलो।',
  'vo7_kabadi':    'और कबाड़ीवाले ने बोला — पूरी तिजोरी के चालीस रुपये।',
  'vo8_finale':    'हँसी ख़त्म। अब अपना ड्रॉर खोलो।',
}
# cast: (voice, rate, pitch, room_available_seconds)
CAST = {
  'vo1_pov':       ('hi-IN-MadhurNeural', '+6%',  '+2Hz', 3.3),
  'vo2_mummy':     ('hi-IN-SwaraNeural',  '+4%',  '+0Hz', 3.6),
  'vo3_narrator1': ('hi-IN-MadhurNeural', '+0%',  '-2Hz', 7.6),
  'vo4_papa':      ('hi-IN-MadhurNeural', '+2%',  '-1Hz', 4.9),
  'vo5_narrator2': ('hi-IN-MadhurNeural', '+2%',  '-2Hz', 5.7),
  'vo6_calc':      ('hi-IN-MadhurNeural', '+0%',  '-1Hz', 10.4),
  'vo7_kabadi':    ('hi-IN-MadhurNeural', '+8%',  '-4Hz', 6.6),
  'vo8_finale':    ('hi-IN-MadhurNeural', '-2%',  '-1Hz', 4.6),
}
async def main():
    for name, text in LINES.items():
        v, r, p, _ = CAST[name]
        await edge_tts.Communicate(text, v, rate=r, pitch=p).save(f'/content/vo_out/{name}.mp3')
        print('  ✓', name)
asyncio.run(main())

from mutagen.mp3 import MP3
print('\n=== FIT CHECK (duration must be <= room +0.3) ===')
ok = True
for name in LINES:
    d = MP3(f'/content/vo_out/{name}.mp3').info.length
    room = CAST[name][3]
    good = d <= room + 0.3
    ok = ok and good
    print(f"  {name}: {d:4.2f}s / room {room}s -> {'✅' if good else '⚠️ SEND ANYWAY, agent will fit'}")
print('\nALL FIT ✅ — run Cell C' if ok else '\nSEND ANYWAY — agent mixes & trims.')


In [ ]:
# ===== CELL B (optional) — pure open-source Piper, native Hindi =====
!pip -q install piper-tts mutagen
import os, urllib.request, shutil
def fetch(url, dst):
    if os.path.exists(dst) and os.path.getsize(dst) > 500_000: return dst
    with urllib.request.urlopen(url) as r, open(dst, 'wb') as f: shutil.copyfileobj(r, f)
    return dst
BASE = 'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/hi/hi_IN/{v}/medium/hi_IN-{v}-medium.onnx'
CFG  = BASE + '.json?download=true.json'
for v in ('pratham', 'priyamvada'):
    fetch(BASE.format(v=v), f'/content/{v}.onnx'); fetch(CFG.format(v=v), f'/content/{v}.onnx.json')
from piper import PiperVoice
VP = PiperVoice.load('/content/pratham.onnx', config_path='/content/pratham.onnx.json')
VF = PiperVoice.load('/content/priyamvada.onnx', config_path='/content/priyamvada.onnx.json')
PCAST = {'vo1_pov':(VP,1.00),'vo2_mummy':(VF,1.02),'vo3_narrator1':(VP,1.12),'vo4_papa':(VP,1.04),
         'vo5_narrator2':(VP,1.10),'vo6_calc':(VP,1.04),'vo7_kabadi':(VP,0.94),'vo8_finale':(VP,1.10)}
for name, text in LINES.items():
    v, ls = PCAST[name]
    try:
        from piper import SynthesisConfig
    except ImportError:
        from piper.config import SynthesisConfig
    with open(f'/content/vo_out/{name}_piper.wav', 'wb') as wf:
        v.synthesize_wav(text, wf, syn_config=SynthesisConfig(length_scale=ls))
    print('  ✓', name)
print('Piper variants saved alongside (name_piper.wav).')

In [ ]:
# ===== CELL C — zip + download =====
import shutil
from google.colab import files
shutil.make_archive('/content/vo_out', 'zip', '/content/vo_out')
files.download('/content/vo_out.zip')